In [1]:
%run Latex_macros.ipynb

<IPython.core.display.Latex object>

# Session 4: Deep Q-learning, on-policy vs off-policy, and SARSA

## Where this session fits

- **Series goal**
    - choose good actions by trial and error
    - from small tables to robots and language models
- **Position**
    - session 4: last session introduced Q-learning with a table
    - this session first scales it to large state spaces with a neural network (deep Q-learning)
    - then asks *which policy an update learns about*
    - the next sessions learn the policy directly (policy gradient)
- **This session's goal**
    - scale Q-learning with a network, and separate the policy that acts from the policy that is learned
- **By the end you can**
    - describe deep Q-learning: replay memory and a lagged target network
    - name the behavior policy and the target policy in an update
    - say why Q-learning is off-policy and SARSA is on-policy
    - explain why they choose different paths on the cliff


## Deep Q-learning (DQN)

The method for Q-learning presented involved creating a *table* implementing the mapping $\actvalfun$.

This is only practical when the size of the table is small
- the number of states for many problems (e.g., games) is extremely large
- not practical

*Deep Q-learning*
- replaces the table by a neural network $\actvalfun_\theta$
    - input: the state $\state$ (for Atari: a stack of recent screen frames)
    - output: one value $\actvalfun_\theta(\state, \act)$ for each action $\act$, so the actions must be *discrete*
    - a convolutional network of this kind reached human-level scores on many of 49 Atari games

The network is **trained** so that its output matches the Q-learning *target*
- $y = \rewseq_{\tt+1} + \disc \max{\act'} \actvalfun_{\theta^-}(\stateseq_{\tt+1}, \act')$, and $y = \rewseq_{\tt+1}$ if the next state is terminal
- the target is a bootstrapped estimate, not a known true value; it is held constant (the gradient flows only through $\actvalfun_\theta$)
- loss: the mean squared error between $y$ and $\actvalfun_\theta(\stateseq_\tt, \actseq_\tt)$


Training uses
- a **mini-batch** of stored transitions
- the gradient of the batch loss, which updates the parameters $\theta$

Two additions to ordinary Q-learning make this work
- a **replay memory** of past transitions
- a **lagged target network** $\theta^-$ (explained below)


The **replay memory** stores transitions as they happen
- mini-batches are drawn at random from it

This **experience replay**
- breaks the strong correlation between consecutive transitions
- reuses each transition many times
- averages the training data over many past behaviors, which smooths learning and helps the network not to forget older experience


**The deep Q-learning loop**

<img src="images/dqn_loop.png" width=90%>

- the main network acts and is trained; the target network only provides $y$
- every $C$ steps the target network is refreshed with the main weights


**Pseudo code for Deep Q-learning**

        Initialize replay memory D to capacity N
        Initialize main Q-network with random weights θ
        Initialize target Q-network with weights θ^- = θ
        
        Set exploration rate ε (annealed from 1 to a small value), discount factor γ, learning rate α (for the optimizer)
        
        For episode = 1 to M:
            Initialize state s
            
            For each step in the episode (until s is terminal):
                With probability ε:
                    Choose random action a (exploration)
                Else:
                    Choose a = argmax_a Q(s, a; θ) (exploitation)

                Take action a, observe reward r and next state s'
                
                Store transition (s, a, r, s') in replay memory D

                Sample random mini-batch of transitions (s_j, a_j, r_j, s'_j) from D   (once D holds enough transitions)

                For each sample in the batch:
                    y_j = r_j                                    if s'_j is terminal
                    y_j = r_j + γ * max_{a'} Q(s'_j, a'; θ^-)   # otherwise (target Q-value)
                    # Predicted Q-value
                    Q_pred = Q(s_j, a_j; θ)

                Compute loss = mean squared error between y_j and Q_pred

                Update network weights θ by minimizing loss via gradient descent

                Every C environment steps:
                    Update target network weights θ^- = θ

                Update state s = s'


Key aspects of the above code:

- $\epsilon$-greedy choice of action

- Replay memory D
    - stores experiences generated by the policy that was in effect when each was created (this matters for *off-policy* learning, explained next)
        - not necessarily the current policy (different values of parameter $\theta$)
    - a batch of experiences is sampled from D to train the NN computing `Q_pred`
- Target `y_j` is the Q-learning target, computed with the lagged weights
- `Q_pred` is the NN current approximation
- in the original Atari DQN: replay memory $N = 10^6$ transitions, target network refreshed every $C = 10^4$ updates, $\epsilon$ annealed from $1$ to $0.1$


- Loss function: MSE between `Q_pred` and calculated `y_j`

        Compute loss = mean squared error between y_j and Q_pred

    - `Q_pred` is the value predicted by the NN
        - for a *batch* of examples
    
    - `y_j` is the "target value":
        - for the batch
        - the value the network is trained to match (a bootstrapped estimate, not a known truth)
        - *defined* by the same calculation as regular Q-learning (Bellman equation)
    
        r + gamma * max_{a'} Q(s', a')
        
        but `Q(s', a')` replaced by NN calculated `Q(s', a'; θ^-)` 
        - where θ^- are the NN's lagged weights
    

A mini-batch by numbers ($\disc = 0.9$, two transitions)

| | reward | next state | $\max{\act'} \actvalfun_{\theta^-}$ at the next state | target $y$ | prediction $\actvalfun_\theta(\state, \act)$ |
|:--|:-:|:--|:-:|:-:|:-:|
| 1 | $+1$ | terminal | (none) | $1$ | $0.6$ |
| 2 | $0$ | not terminal | $0.8$ | $0 + 0.9 \times 0.8 = 0.72$ | $0.5$ |

- loss $= \frac{1}{2}\left[(1 - 0.6)^2 + (0.72 - 0.5)^2\right] \approx 0.10$
- one gradient step on $\theta$ lowers this loss; $y$ is held fixed


Why the lagged weights $\theta^-$ in computing the target?

The target $y$ is computed from the network we are training
- with the *current* weights, every gradient step also moves the target
- the network chases its own output: a moving target as well as a moving function
- this can oscillate or diverge

The lagged copy $\theta^-$ is refreshed only every $C$ steps
- so for a while the target holds still


# On-policy versus Off-policy

Every update involves two policies
- the **behavior policy** chooses the actions that generate the data
    - for Q-learning: $\epsilon$-greedy with respect to the current $\actvalfun$
- the **target policy** is the policy whose value the update learns about
    - for Q-learning: the *greedy* policy, because the target uses $\max{\act'}$

The *target value* $y$ is a number: the update's estimate of the value of the greedy continuation; it is not the policy itself.

Q-learning's table converges to $\actvalfun_*$, the value of the greedy (optimal) policy.


When the Behavior and Target policies are identical
- we call the method *On-policy*

If the Behavior and Target policies are potentially different
- we call the method *Off-policy*

Q-learning is **off-policy**

The main reason
- the target uses the *greedy* action (the max), while the actions that generated the data came from the $\epsilon$-greedy behavior policy
- this holds even for the tabular algorithm, with no replay
- no correction for the difference between the policies (importance weights) is needed: $\actvalfun(\state, \act)$ conditions on the *first* action, and the max handles all later ones

Experience replay adds a second source
- stored transitions were generated by *older* versions of the behavior policy (replay is possible *because* Q-learning is off-policy)

The lagged network $\theta^-$ is not a policy: it only stabilizes the target.

| Method | Behavior policy | Target policy |
|:--|:--|:--|
| Q-learning, DQN | $\epsilon$-greedy | greedy |
| SARSA | $\epsilon$-greedy | the same $\epsilon$-greedy policy |


# SARSA: the on-policy counterpart

SARSA is a small change to the Q-learning update
- Q-learning target: $\rewseq_{\tt+1} + \disc \max{\act'} \actvalfun(\stateseq_{\tt+1}, \act')$
- SARSA target: $\rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1})$
    - $\actseq_{\tt+1}$ is the action the behavior policy *actually chooses* next
$$
\actvalfun(\stateseq_\tt, \actseq_\tt) \leftarrow \actvalfun(\stateseq_\tt, \actseq_\tt) + \alpha \bigl[ \rewseq_{\tt+1} + \disc \, \actvalfun(\stateseq_{\tt+1}, \actseq_{\tt+1}) - \actvalfun(\stateseq_\tt, \actseq_\tt) \bigr]
$$

The name comes from the quintuple the update uses: $\stateseq_\tt, \actseq_\tt, \rewseq_{\tt+1}, \stateseq_{\tt+1}, \actseq_{\tt+1}$
- the exploratory choices are *inside* the values SARSA learns
- so it learns the value of the policy it follows: *on-policy*

        Choose a from s with the epsilon-greedy policy
        Repeat for each step:
            Take a, observe r and s'
            Choose a' from s' with the epsilon-greedy policy
            Q(s, a) = Q(s, a) + alpha * [r + gamma * Q(s', a') - Q(s, a)]     (Q = 0 at terminal states)
            s = s'; a = a'


One transition, two targets (illustrative numbers)
- the agent walks along the edge of the cliff and, after a step with reward $\rewseq = -1$, lands in $\state'$, also on the edge
- the values there: $\actvalfun(\state', \text{right}) = -8$ (keep walking) and $\actvalfun(\state', \text{down}) = -100$ (into the cliff); $\disc = 1$

What are the Q-learning and the SARSA targets if the behavior policy's next action is a slip into the cliff?


| the next action chosen by the behavior policy | Q-learning target | SARSA target |
|:--|:-:|:-:|
| right (the greedy action) | $-1 + (-8) = -9$ | $-1 + (-8) = -9$ |
| down, into the cliff (an exploratory slip) | $-1 + (-8) = -9$ | $-1 + (-100) = -101$ |

Q-learning ignores the slip; SARSA includes it, so the values along the edge drop.


Q-learning and SARSA therefore learn *different things*
- Q-learning learns the value of the **greedy** policy, even though it acts $\epsilon$-greedily
- SARSA learns the value of the **$\epsilon$-greedy** policy it actually follows, exploratory mistakes included

When exploration is harmless, both end up with similar values
When exploration is *costly* (a cliff), the values differ, and so do the paths

In the tabular case
- Q-learning converges to $\actvalfun_*$ for any fixed $\epsilon > 0$, if every state-action pair keeps being visited and the step size decays suitably
- SARSA converges to $\actvalfun_*$ only if $\epsilon \to 0$ as well
- the cliff contrast above is for a *fixed* $\epsilon$ (the animation's $\epsilon$ only decays to $0.01$)


## Grid-world: SARSA vs Q-learning on the cliff
    
We need to navigate in a $(4 \times 12)$ grid
- from start cell $S$
- to goal cell $G$
- avoiding the cliff cells: stepping on one costs $-100$ (instead of $-1$), sends the agent back to $S$, and the episode continues

Both learners here use tables (no neural network).

There is a negative reward ($-1$) for each time step so the reward is maximized by getting to the goal quickly.

There are four moves (up, right, down, left); the episode ends at $G$. Both learners explore $\epsilon$-greedily.

Q-learning learns the values of the **greedy** policy
- its best path hugs the edge: the shortest route
- but while it explores ($\epsilon$-greedy) it sometimes steps off the cliff

SARSA learns the values of the **$\epsilon$-greedy** policy it follows
- the edge is dangerous *for that policy*, so SARSA prefers a longer path away from the edge



<img src="images/cliff_sarsa_vs_qlearning.png" width=100%>

- left: Q-learning's greedy path is the 13-step edge route in all 100 runs; SARSA's usually has 17 steps (77 of 100 runs; the others learn a 19-step path or a route that is not finished)
- right: while exploring ($\epsilon = 0.1$), Q-learning earns *less* reward per episode, because it falls off the cliff
- setting: $\alpha = 0.5$, $\disc = 1$, $500$ episodes, $100$ runs, fixed $\epsilon$


**Animation** (it loops): the greedy paths after training, drawn step by step
- Q-learning (orange) hugs the cliff; SARSA (blue) goes around
- settings: $\alpha = 0.2$, $\disc = 0.99$, $\epsilon$ decaying to $0.01$, $5000$ episodes

<img src="images/cliffwalking_paths_qlearning.gif" width=80%>


Why the paths differ
- SARSA's target uses the next action the behavior policy actually takes
    - near the cliff an exploratory step can fall off, and the large negative reward lowers the values that lead there
    - the lowered values spread to the states along the edge, so SARSA's greedy path moves away from the cliff
- Q-learning's target uses the max over the next actions
    - it ignores the exploratory step in the target, so the edge keeps its high value
    - its greedy path is the best one for an agent that stops exploring, but the exploring agent falls more often

Neither is wrong: they answer different questions
- Q-learning: what is the best path if I stop exploring?
- SARSA: what is the best path given that I keep exploring?


## SARSA vs Q-learning: Summary

| Feature | SARSA (on-policy) | Q-learning / DQN (off-policy) |
|:--|:--|:--|
| Policy learned about | the $\epsilon$-greedy behavior policy | the greedy policy |
| Update target | $\rewseq + \disc\,\actvalfun(\state', \act')$ (the action actually taken next) | $\rewseq + \disc \max{\act'} \actvalfun(\state', \act')$ (the best next action) |
| Exploration risk | included in the values (safer paths) | ignored in the values (shortest path, riskier while exploring) |
| Experience replay | does not fit naturally: stored transitions came from older policies | fits: the one-step target does not use the next action actually taken, so older data can be reused |
| Function approximation | works (e.g. deep SARSA) | works (DQN); function approximation + bootstrapping + off-policy data (the "deadly triad") can be unstable, hence replay and the target network |


# Learning via Code

[Here](RL_OnPolicy_vs_OffPolicy_code_examples.ipynb) is a notebook that demonstrates
- Q-Learning
    - with and without replay memory
- SARSA

## How Each Example Highlights a Salient Point

1. **Q-learning (Off-policy):**
    - **Purpose:** Demonstrates how the behavior policy (epsilon-greedy; explores randomly) differs from the target policy (greedy; always chooses highest Q-value).
    - **Salient Point:** Shows that updates use the greedy maximum Q-value (off-policy), even when the sampled action was exploratory.
2. **SARSA (On-policy):**
    - **Purpose:** Demonstrates on-policy learning where both the behavior and target policies are epsilon-greedy and always match.
    - **Salient Point:** The Q update uses the value of the exact next action chosen by the behavior policy, making sampling and updating fully aligned.
3. **Q-learning with Experience Replay Buffer:**
    - **Purpose:** Shows how experience transitions are stored in a replay buffer and reused for updates.
    - **Salient Point:** Illustrates how off-policy Q-learning leverages exploratory samples and batch updates from the buffer to improve sample efficiency and stabilize learning.


# What we established, and what it sets up

- **Established**
    - deep Q-learning replaces the table with a network, using replay memory and a lagged target network
    - the behavior policy chooses actions; the target policy is the one an update learns about
    - Q-learning learns the greedy policy (off-policy); SARSA learns the policy it follows (on-policy)
    - on the cliff they learn different values, hence different paths
- **Sets up**
    - next: learning the policy directly with policy gradients, where the on-/off-policy question returns
    - later: bias and variance of these estimators


In [4]:
print("Done")

Done
